In [ ]:
# %% Setup
from pyspark.sql import SparkSession, functions as F

from yelp_sentiment.config import settings
from yelp_sentiment.ingestion.schemas import BUSINESS_SCHEMA, REVIEW_SCHEMA, USER_SCHEMA

spark = (SparkSession.builder.appName("yelp-explore")
         .config("spark.driver.memory", "4g").getOrCreate())

raw_dir = settings.data_dir / "raw"
def raw(name: str) -> str:
    return str(raw_dir / f"yelp_academic_dataset_{name}.json")

reviews = spark.read.schema(REVIEW_SCHEMA).json(raw("review"))
business = spark.read.schema(BUSINESS_SCHEMA).json(raw("business"))
users = spark.read.schema(USER_SCHEMA).json(raw("user"))

print("reviews", reviews.count(), "business", business.count(), "users", users.count())
reviews.groupBy("stars").count().orderBy("stars").show()
reviews.select(F.length("text").alias("len")).summary("min", "25%", "50%", "75%", "95%",
                                                      "max").show()

# Intégrité : avis orphelins
print("orphelins business:", reviews.join(business, "business_id", "left_anti").count())
print("orphelins user:", reviews.join(users, "user_id", "left_anti").count())

# Nulls sur les champs critiques
reviews.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
                for c in ["review_id", "business_id", "user_id", "stars", "text"]]).show()

# Part des restaurants
resto = business.filter(F.col("categories").contains("Restaurants"))
print("business restaurants:", resto.count())
print("avis restaurants:", reviews.join(resto.select("business_id"), "business_id").count())

# Vérifier le bug "20,20" dans elite
users.filter(F.col("elite").contains("20,20")).select("elite").show(5, truncate=False)

reviews 6990280 business 150346 users 1987897


+-----+-------+
|stars|  count|
+-----+-------+
|  1.0|1069561|
|  2.0| 544240|
|  3.0| 691934|
|  4.0|1452918|
|  5.0|3231627|
+-----+-------+



+-------+----+
|summary| len|
+-------+----+
|    min|   1|
|    25%| 229|
|    50%| 406|
|    75%| 720|
|    95%|1564|
|    max|5000|
+-------+----+



orphelins business: 0


orphelins user: 33


+---------+-----------+-------+-----+----+
|review_id|business_id|user_id|stars|text|
+---------+-----------+-------+-----+----+
|        0|          0|      0|    0|   0|
+---------+-----------+-------+-----+----+

business restaurants: 52268


avis restaurants: 4724471
+--------------------------------------------------------------------------------+
|elite                                                                           |
+--------------------------------------------------------------------------------+
|2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,20,20,2021               |
|2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,20,20,2021                    |
|2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,20,20,2021                    |
|2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,20,20,2021|
|2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,20,20,2021                    |
+--------------------------------------------------------------------------------+
only showing top 5 rows
